# TerraSpectra — Day 3: Dataset Exploration & Spectral Signatures

## 1. Objective & Prerequisites
- **Objective**: Perform numerical health audits, plot representative spectral signatures across agricultural classes, and generate a false-color composite.
- **Prerequisites**: Day 2 ingestion completed; `terraspectra.data.exploration` module installed.

## 2. Core Concepts
- **Spectral Signature**: Every material reflects solar radiation uniquely across different wavelengths. Healthy vegetation reflects strongly in the Near-Infrared (NIR) spectrum while absorbing Red light (due to chlorophyll absorption).
- **False-Color Composite**: Since human vision is limited to RGB (400–700 nm), remote sensing combines non-visible bands (e.g. NIR to Red, Red to Green, Green to Blue) to highlight vegetation vitality.

In [1]:
# Import modules
import matplotlib.pyplot as plt
import numpy as np
from terraspectra.data.loader import SalinasLoader
from terraspectra.data.exploration import HyperspectralExplorer

# Load dataset
loader = SalinasLoader(raw_data_dir='../data/raw')
dataset = loader.load()
explorer = HyperspectralExplorer(dataset=dataset, output_dir='../outputs')

In [2]:
# 1. Audit Numerical Quality
quality = explorer.audit_numerical_quality()
print('--- Numerical Health Audit ---')
print(f'Total Pixels:  {quality.total_pixels:,}')
print(f'NaN Count:     {quality.nan_count}')
print(f'Inf Count:     {quality.inf_count}')
print(f'Value Range:   [{quality.min_value}, {quality.max_value}]')
print(f'Mean ± Std:    {quality.mean_value:.2f} ± {quality.std_value:.2f}')
print(f'Health Status: {"ALL FINITE (CLEAN)" if quality.is_finite else "CORRUPT"}')

--- Numerical Health Audit ---
Total Pixels:  111,104
NaN Count:     0
Inf Count:     0
Value Range:   [-11.0, 9207.0]
Mean ± Std:    1196.40 ± 1098.40
Health Status: ALL FINITE (CLEAN)


In [3]:
# 2. Render False-Color Composite (Bands 45: NIR, 25: Red, 15: Green)
rgb_stretched = explorer.generate_false_color_composite(
    bands=(45, 25, 15),
    save_path='../outputs/figures/salinas_false_color_composite.png'
)

plt.figure(figsize=(7, 10))
plt.imshow(rgb_stretched)
plt.title('Salinas False-Color Composite (NIR: 45, Red: 25, Green: 15)\n[Array Pixel Coordinates]')
plt.xlabel('Column Index (0 to 216)')
plt.ylabel('Row Index (0 to 511)')
plt.show()

C:\Users\abhis\AppData\Local\Temp\ipykernel_23368\3538946916.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Observation on False-Color Composite
- In standard false-color (Color Infrared), healthy crops appear in vibrant shades of red and magenta because vegetation reflectance peaks sharply in the NIR band (band 45, mapped to the Red channel).
- Fallow fields and bare soil appear in muted brownish-cyan tones.

In [4]:
# 3. Extract and Plot Representative Spectral Signatures for 4 Contrasting Crops
contrasting_classes = [1, 3, 7, 8]  # Broccoli, Fallow, Celery, Grapes
signatures = explorer.generate_spectral_signatures(
    classes_to_plot=contrasting_classes,
    save_path='../outputs/figures/salinas_contrasting_signatures.png'
)

plt.figure(figsize=(10, 5))
for cid in contrasting_classes:
    cname = signatures[cid]['class_name']
    mean_vals = np.array(signatures[cid]['mean'])
    std_vals = np.array(signatures[cid]['std'])
    bands_idx = np.arange(len(mean_vals))
    plt.plot(bands_idx, mean_vals, label=f'Class {cid}: {cname}', linewidth=2)
    plt.fill_between(bands_idx, mean_vals - std_vals, mean_vals + std_vals, alpha=0.15)

plt.title('Contrasting Spectral Signatures (Mean ± 1 SD)')
plt.xlabel('Spectral Band Index (0 to 203)')
plt.ylabel('Calibrated Radiance Value')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

C:\Users\abhis\AppData\Local\Temp\ipykernel_23368\2866908156.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. What I Learned & Common Mistakes
1. **The "Red Edge" Transition**: Notice the steep slope around band indices 30–50 across vegetation classes. This marks the transition from chlorophyll absorption in the red wavelengths to cell structure scattering in the NIR.
2. **Percentile Stretching**: Raw radiance values have large dynamic ranges. Linear 2% to 98% percentile clipping is necessary to reveal details without washing out bright soil pixels.
3. **Geospatial Integrity**: The exploration report confirms no EPSG/CRS headers exist in the Salinas `.mat`. Spatial coordinates must strictly be described as row/column array indices.

### Interview Question
*Q: Why do spectral signature plots show dips in certain bands?*
*A: Even in corrected scenes, narrow residual atmospheric absorption features and specific biochemical bonds (like water, lignin, cellulose) absorb specific electromagnetic wavelengths, creating diagnostic dips (absorption features).*

## 4. Verification & Milestone Status
- [x] Verified zero NaNs and zero Infs in the entire cube.
- [x] Successfully plotted and saved false-color composite image.
- [x] Generated representative spectral curves with variance bands.
- [x] Documented spatial array coordinate system without fabricated georeferencing.